# End-to-End Fraud Detection with FinCrime-ML

**Author:** Temidayo Akindahunsi  
**Domain:** UK retail card fraud detection  
**Regulatory context:** FCA PROD 2.1, PRA SS1/23, SR 11-7

---

## Overview

This notebook walks through the complete fraud detection pipeline implemented in `fincrime_ml.fraud`. It is structured as a model development workflow a practitioner would follow in a UK-regulated financial institution, from synthetic data generation through to threshold selection and regulatory documentation artefacts.

The pipeline covers seven stages:

1. **Synthetic data generation** — realistic UK card transaction data with injected fraud typologies
2. **Feature engineering** — velocity windows, amount deviation, MCC risk encoding
3. **Class imbalance handling** — SMOTE versus cost-sensitive weighting benchmark
4. **Model training** — XGBoost champion with logistic regression challenger
5. **SHAP explainability** — per-prediction reason codes and feature importance
6. **Evaluation** — threshold analysis, monetary cost matrix, champion/challenger comparison
7. **Threshold selection** — operationally justified decision with documented rationale

### Regulatory note

PRA SS1/23 Section 5 requires that model validation includes out-of-sample performance testing with documented metrics and threshold sensitivity analysis. FCA PROD 2.1 (treating customers fairly) requires that automated decline decisions consider the impact on legitimate customers. The cost matrix in Section 6 provides the quantitative basis for both requirements.

## 0. Environment setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# FinCrime-ML imports
from fincrime_ml.core.data.synth_cards import SyntheticTransactionGenerator
from fincrime_ml.fraud.evaluation import FraudEvaluator
from fincrime_ml.fraud.explain import FraudExplainer
from fincrime_ml.fraud.features import FraudFeatureEngineer
from fincrime_ml.fraud.imbalance import ImbalanceHandler
from fincrime_ml.fraud.models.logistic_baseline import LogisticFraudBaseline
from fincrime_ml.fraud.models.xgb_classifier import FEATURE_COLS, XGBFraudClassifier

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42
print("Environment ready.")

---
## 1. Synthetic data generation

Real fraud datasets are either proprietary or heavily anonymised (the IEEE-CIS dataset strips merchant names and card numbers entirely). For development and CI we use `SyntheticTransactionGenerator`, which produces UK-flavoured card transaction data with realistic fraud typology injection: card-not-present (CNP), account takeover (ATO), and bust-out patterns.

The generator controls:
- **Fraud rate:** UK Finance 2023 reports card fraud at approximately 0.07% of transaction volume; we use a higher rate (8%) in development data to give the model sufficient positive examples for training
- **Account velocity:** genuine accounts have plausible transaction cadence; fraudulent accounts show burst patterns
- **Amount distribution:** fraud transactions skew toward the upper tail of the legitimate distribution, mirroring the pattern observed in CNP fraud

> **Data governance note:** In production, this module is replaced by a feed from the core banking system. The synthetic generator is used exclusively in development, testing, and model validation on non-live data.

In [ ]:
gen = SyntheticTransactionGenerator(n_accounts=500, seed=SEED)
df = gen.generate(n_transactions=5_000, fraud_rate=0.08)

print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {df['is_fraud'].mean():.2%}")
print(f"\nColumn list: {list(df.columns)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Amount distribution by fraud label
for label, grp in df.groupby("is_fraud"):
    axes[0].hist(
        grp["amount"].clip(upper=500),
        bins=40,
        alpha=0.6,
        label="Fraud" if label else "Legitimate",
        density=True,
    )
axes[0].set_xlabel("Transaction amount (GBP, capped at 500)")
axes[0].set_ylabel("Density")
axes[0].set_title("Amount distribution by label")
axes[0].legend()

# Class balance
counts = df["is_fraud"].value_counts()
axes[1].bar(["Legitimate", "Fraud"], counts.values, color=["steelblue", "firebrick"])
axes[1].set_title("Class balance")
axes[1].set_ylabel("Transaction count")
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 20, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 2. Feature engineering

`FraudFeatureEngineer` builds the 16-column feature matrix used by all downstream models. The features are grouped into three families:

**Velocity features** capture abnormal transaction cadence per account:
- `txn_count_1h`, `txn_count_24h`, `txn_count_168h`: rolling transaction counts over 1 hour, 24 hours, and 7 days
- `amount_sum_1h`, `amount_sum_24h`, `amount_sum_168h`: rolling spend totals over the same windows

Velocity features are among the strongest fraud signals in card data. A burst of 6 transactions within one hour from the same account card is highly anomalous for most customer segments, even if each individual transaction amount appears normal.

**Amount deviation:** `amount_zscore` measures how far the transaction amount deviates from the account's historical mean, normalised by the account's standard deviation. A z-score above 3.0 indicates the amount is more than 3 standard deviations above the account's typical spend.

**MCC risk encoding:** `mcc_risk_score` maps the merchant category code to a risk tier based on known high-risk merchant categories (online gambling, foreign exchange, digital goods). This is a common practitioner feature derived from the MCC taxonomy maintained by card schemes.

**Hour and day-of-week:** `hour_of_day` and `day_of_week` capture temporal patterns. Fraud transactions are disproportionately concentrated between midnight and 04:00 when genuine cardholder disputes are rare.

In [ ]:
engineer = FraudFeatureEngineer()
df_features = engineer.transform(df)

print(f"Feature columns ({len(FEATURE_COLS)}):")
for col in FEATURE_COLS:
    print(f"  {col}")

In [ ]:
# Feature correlation with fraud label
correlations = (
    df_features[FEATURE_COLS + ["is_fraud"]]
    .corr()["is_fraud"]
    .drop("is_fraud")
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["firebrick" if v > 0 else "steelblue" for v in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson correlation with is_fraud")
ax.set_title("Feature-label correlations")
plt.tight_layout()
plt.show()

print("\nTop 5 positive correlates:")
print(correlations.head())

---
## 3. Class imbalance

Fraud datasets are inherently imbalanced. In production card portfolios, fraud rates are typically below 0.1%. Even at our synthetic 8% rate, legitimate transactions outnumber fraud transactions 11:1. A naive classifier that predicts every transaction as legitimate achieves 92% accuracy but catches zero fraud: this is why accuracy is the wrong optimisation target for fraud models. We use AUC-PR (area under the precision-recall curve) as the primary metric, which is sensitive to the minority class.

Two approaches to class imbalance are standard in practice:

**SMOTE (Synthetic Minority Oversampling Technique):** generates synthetic fraud examples by interpolating between existing fraud transactions in feature space. The risk is that synthetic samples may not reflect real fraud patterns, particularly for non-convex fraud clusters in feature space.

**Cost-sensitive weighting:** directly penalises the model more heavily for misclassifying fraud, without generating synthetic data. XGBoost's `scale_pos_weight` parameter implements this. The weight is typically set to `n_legitimate / n_fraud`.

`ImbalanceHandler.benchmark()` evaluates both approaches under cross-validation so the practitioner can make an evidence-based choice for their specific dataset.

In [ ]:
X_raw = df_features[FEATURE_COLS].to_numpy(dtype=float)
y = df["is_fraud"].to_numpy(dtype=int)

handler = ImbalanceHandler(seed=SEED, cv_folds=3)
benchmark_results = handler.benchmark(X_raw, y)

results_df = pd.DataFrame(
    [
        {
            "strategy": r.strategy,
            "mean_auc_pr": round(r.mean_auc_pr, 4),
            "std_auc_pr": round(r.std_auc_pr, 4),
            "mean_roc_auc": round(r.mean_roc_auc, 4),
        }
        for r in benchmark_results
    ]
)
print(results_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    results_df["strategy"],
    results_df["mean_auc_pr"],
    yerr=results_df["std_auc_pr"],
    capsize=5,
    color=["steelblue", "darkorange"],
    alpha=0.85,
)
ax.set_ylabel("Mean AUC-PR (3-fold CV)")
ax.set_title("SMOTE vs cost-sensitive: AUC-PR comparison")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---
## 4. Model training

### 4.1 Train/test split

We use a stratified 80/20 holdout. In a time-series-aware pipeline this would be a temporal split (train on months 1-9, test on months 10-12); with synthetic data we use random stratified splitting as a proxy.

### 4.2 XGBoost champion

`XGBFraudClassifier` is the primary production model. Key design choices:

- `eval_metric=aucpr`: directly optimises the area under the precision-recall curve, the primary business metric
- `scale_pos_weight`: cost-sensitive weighting; set to the class ratio to penalise false negatives more heavily
- Stratified k-fold CV: ensures AUC-PR estimates are stable across different fraud prevalence samples
- Early stopping on a held-out eval set: prevents overfitting to the training fold

### 4.3 Logistic regression challenger

`LogisticFraudBaseline` serves three functions: performance floor, regulatory interpretability, and feature importance cross-validation. If the XGBoost model does not materially outperform logistic regression on AUC-PR, the problem is in the features or data, not the model architecture.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df, test_size=0.20, stratify=df["is_fraud"], random_state=SEED
)
print(f"Train: {len(df_train):,} rows, fraud rate: {df_train['is_fraud'].mean():.2%}")
print(f"Test:  {len(df_test):,} rows, fraud rate: {df_test['is_fraud'].mean():.2%}")

In [ ]:
print("Training XGBoost champion...")
xgb_clf = XGBFraudClassifier(n_cv_folds=3, seed=SEED)
xgb_clf.train(df_train)

cv_xgb = xgb_clf.cv_summary()
print(f"\nXGBoost CV AUC-PR: {cv_xgb['auc_pr'].mean():.4f} (+/- {cv_xgb['auc_pr'].std():.4f})")
print(f"XGBoost CV ROC-AUC: {cv_xgb['roc_auc'].mean():.4f}")

In [ ]:
print("Training logistic regression baseline...")
lr_clf = LogisticFraudBaseline(n_cv_folds=3, seed=SEED)
lr_clf.train(df_train)

cv_lr = lr_clf.cv_summary()
print(f"\nLogistic CV AUC-PR: {cv_lr['auc_pr'].mean():.4f} (+/- {cv_lr['auc_pr'].std():.4f})")
print(f"Logistic CV ROC-AUC: {cv_lr['roc_auc'].mean():.4f}")

In [ ]:
# Cross-validation comparison
fig, ax = plt.subplots(figsize=(7, 4))

folds = cv_xgb["fold"].values
x = np.arange(len(folds))
width = 0.35

ax.bar(x - width / 2, cv_xgb["auc_pr"], width, label="XGBoost", color="steelblue")
ax.bar(x + width / 2, cv_lr["auc_pr"], width, label="Logistic", color="darkorange")

ax.set_xticks(x)
ax.set_xticklabels([f"Fold {f}" for f in folds])
ax.set_ylabel("AUC-PR")
ax.set_title("Per-fold AUC-PR: XGBoost vs Logistic Regression")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---
## 5. SHAP explainability

Explainability is a regulatory requirement in the UK fraud context, not an optional add-on. The FCA expects firms to be able to explain why a specific transaction was declined or flagged. SR 11-7 (the Federal Reserve and OCC model risk guidance widely adopted by UK firms under PRA SS1/23) requires that automated systems produce intelligible explanations for individual decisions.

`FraudExplainer` uses SHAP (SHapley Additive exPlanations) to decompose each model prediction into per-feature contributions. SHAP values have a formal game-theoretic interpretation: the SHAP value for feature $f$ on transaction $i$ is the average marginal contribution of $f$ to the prediction across all possible feature orderings.

Key outputs:
- **Reason codes:** the top N features driving the score for each transaction, ranked by absolute SHAP value
- **Direction:** whether each feature pushes the score toward fraud (`increases_risk`) or away from it (`reduces_risk`)
- **Feature summary:** mean absolute SHAP value per feature across a cohort, used in model validation documentation

> **Analyst note:** A large negative SHAP value (reduces_risk) can be just as important as a large positive one. A transaction flagged primarily because of a high velocity score may still have a low absolute risk score if the amount deviation is strongly negative (i.e. a very small transaction). Reason codes include both directions.

In [ ]:
X_test = xgb_clf.prepare_features(df_test).to_numpy(dtype=float)

explainer = FraudExplainer(xgb_clf.model, feature_names=FEATURE_COLS)
explainer.fit()

summary = explainer.feature_summary(X_test)
print("Feature importance (mean absolute SHAP value on test set):")
print(summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    summary["feature"][::-1],
    summary["mean_abs_shap"][::-1],
    color="steelblue",
)
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("XGBoost feature importance (SHAP)")
plt.tight_layout()
plt.show()

In [ ]:
# Reason codes for the first 5 flagged transactions in the test set
xgb_scores_test = xgb_clf.predict(df_test)
flagged_idx = xgb_scores_test[xgb_scores_test["risk_score"] > 0.5].index[:5]

if len(flagged_idx) > 0:
    reason_codes = explainer.reason_codes(
        X_test[df_test.index.get_indexer(flagged_idx)],
        transaction_ids=list(flagged_idx),
        top_n=3,
    )
    print("Top 3 reason codes for flagged transactions:")
    print(reason_codes.to_string(index=False))
else:
    print("No transactions scored above 0.5 at this threshold.")

In [ ]:
# Single-transaction explanation (audit trail format)
explanation = explainer.explain_single(X_test, idx=0, top_n=5)
print(f"Transaction index: {explanation['transaction_idx']}")
print("\nTop contributing features:")
for feat in explanation["top_features"]:
    direction_symbol = "+" if feat["direction"] == "increases_risk" else "-"
    print(
        f"  [{direction_symbol}] {feat['feature']:<25} "
        f"raw={feat['raw_value']:.4f}  shap={feat['shap_value']:+.4f}  ({feat['direction']})"
    )

---
## 6. Evaluation

### 6.1 Champion/challenger comparison

PRA SS1/23 Section 5.3 requires model validation to include a performance comparison between the candidate model and a simpler benchmark. `FraudEvaluator.compare_models()` computes AUC-PR and ROC-AUC on the held-out test set for both models.

AUC-PR is the primary metric because it directly measures the precision-recall tradeoff across all thresholds. ROC-AUC is reported for completeness but is less informative under class imbalance.

In [ ]:
y_test = df_test["is_fraud"].to_numpy(dtype=int)

# Get probability scores from both models
xgb_scores = xgb_clf.predict(df_test)["risk_score"].to_numpy()
lr_scores = lr_clf.predict(df_test)["risk_score"].to_numpy()

evaluator = FraudEvaluator(fp_cost=15.0, fn_cost=250.0)

comparison = evaluator.compare_models(
    y_test,
    {"XGBoost": xgb_scores, "Logistic": lr_scores},
    threshold=0.5,
)
print("Champion/challenger comparison (holdout test set):")
print(comparison.to_string(index=False))

In [ ]:
# PR curves
pr_xgb = evaluator.pr_curve(y_test, xgb_scores)
pr_lr = evaluator.pr_curve(y_test, lr_scores)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Precision-recall curves
axes[0].plot(
    pr_xgb["recall"], pr_xgb["precision"],
    label=f"XGBoost (AUC-PR={comparison.loc[comparison['model']=='XGBoost', 'auc_pr'].values[0]:.3f})",
    color="steelblue",
)
axes[0].plot(
    pr_lr["recall"], pr_lr["precision"],
    label=f"Logistic (AUC-PR={comparison.loc[comparison['model']=='Logistic', 'auc_pr'].values[0]:.3f})",
    color="darkorange",
    linestyle="--",
)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall curves")
axes[0].legend(loc="upper right")

# ROC curves
roc_xgb = evaluator.roc_curve(y_test, xgb_scores)
roc_lr = evaluator.roc_curve(y_test, lr_scores)
axes[1].plot(
    roc_xgb["fpr"], roc_xgb["tpr"],
    label=f"XGBoost (ROC-AUC={comparison.loc[comparison['model']=='XGBoost', 'roc_auc'].values[0]:.3f})",
    color="steelblue",
)
axes[1].plot(
    roc_lr["fpr"], roc_lr["tpr"],
    label=f"Logistic (ROC-AUC={comparison.loc[comparison['model']=='Logistic', 'roc_auc'].values[0]:.3f})",
    color="darkorange",
    linestyle="--",
)
axes[1].plot([0, 1], [0, 1], "k:", alpha=0.4, label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC curves")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

### 6.2 Threshold analysis

The model outputs a continuous risk score between 0 and 1. A decision threshold converts this to a binary decision: flag or pass. Choosing the threshold is an operational decision that must consider:

- **Alert capacity:** how many alerts can the fraud operations team review per day?
- **False positive tolerance:** what proportion of declined transactions can be legitimate before customer complaints become a regulatory concern?
- **Fraud loss appetite:** what is the acceptable fraud loss per period?

The threshold analysis sweeps 100 values and reports precision, recall, F1, and alert volume at each point.

In [ ]:
threshold_df = evaluator.threshold_analysis(y_test, xgb_scores)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
axes[0, 0].plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
axes[0, 0].plot(threshold_df["threshold"], threshold_df["f1"], label="F1", linestyle="--")
axes[0, 0].set_xlabel("Threshold")
axes[0, 0].set_title("Precision, Recall, F1 vs threshold")
axes[0, 0].legend()

axes[0, 1].plot(threshold_df["threshold"], threshold_df["alert_rate"] * 100, color="firebrick")
axes[0, 1].set_xlabel("Threshold")
axes[0, 1].set_ylabel("Alert rate (%)")
axes[0, 1].set_title("Alert rate vs threshold")

axes[1, 0].plot(threshold_df["threshold"], threshold_df["n_false_positives"], color="darkorange", label="FP")
axes[1, 0].plot(threshold_df["threshold"], threshold_df["n_false_negatives"], color="steelblue", label="FN")
axes[1, 0].set_xlabel("Threshold")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("False positives and false negatives vs threshold")
axes[1, 0].legend()

# Cost curve
fp_costs = threshold_df["n_false_positives"] * evaluator.fp_cost
fn_costs = threshold_df["n_false_negatives"] * evaluator.fn_cost
axes[1, 1].fill_between(threshold_df["threshold"], fp_costs, alpha=0.5, label=f"FP cost (£{evaluator.fp_cost}/case)", color="darkorange")
axes[1, 1].fill_between(threshold_df["threshold"], fn_costs, alpha=0.5, label=f"FN cost (£{evaluator.fn_cost}/case)", color="steelblue")
axes[1, 1].plot(threshold_df["threshold"], fp_costs + fn_costs, color="black", linewidth=1.5, label="Total cost")
axes[1, 1].set_xlabel("Threshold")
axes[1, 1].set_ylabel("Expected cost (GBP)")
axes[1, 1].set_title("Monetary cost vs threshold")
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 6.3 False positive cost matrix

The monetary cost framework makes the business tradeoff explicit. Default parameters:

- **FP cost (£15 per case):** estimated customer friction cost and complaint handling overhead for a wrongly declined legitimate transaction. This is conservative; for high-value customers the cost may be significantly higher due to churn risk.
- **FN cost (£250 per case):** estimated average fraud loss per transaction (UK Finance 2023 average for CNP fraud). This does not include reputational cost or scheme chargeback fees.

The asymmetry (250:15, approximately 17:1) means a single missed fraud transaction costs as much as 17 wrongly declined legitimate transactions. This drives the threshold toward lower values (higher recall) unless alert capacity is constrained.

> **FCA PROD 2.1 note:** Firms must consider the fair treatment of retail customers when setting automated decision thresholds. A threshold that is too low (too many false positives) may disproportionately affect vulnerable customers who have less ability to dispute a declined transaction. The cost matrix should be reviewed alongside customer segment impact assessments.

In [ ]:
# Cost matrix at three operationally relevant thresholds
for t in [0.3, 0.5, 0.7]:
    cm = evaluator.cost_matrix(y_test, xgb_scores, threshold=t)
    print(
        f"Threshold={t:.1f}: "
        f"FP={cm.n_false_positives} (cost £{cm.total_fp_cost:,.0f}), "
        f"FN={cm.n_false_negatives} (cost £{cm.total_fn_cost:,.0f}), "
        f"total=£{cm.total_cost:,.0f}"
    )

---
## 7. Threshold selection

`optimal_threshold()` automates threshold selection under three strategies. For a production deployment the choice of strategy depends on the operational context:

| Strategy | Use case |
|---|---|
| `f1` | Balanced operational context with equal weight on catching fraud and minimising false positives |
| `cost` | Business context where fraud losses and FP handling costs are monetised |
| `recall_at_precision` | Constrained alert queue: maximise fraud catch rate subject to at least 50% of alerts being genuine fraud |

The selected threshold must be documented, justified, and reviewed at model revalidation (minimum annually under PRA SS1/23).

In [ ]:
strategies = ["f1", "cost", "recall_at_precision"]
print("Optimal threshold by strategy (XGBoost, holdout test set):")
print(f"{'Strategy':<25} {'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Alerts':>8}")
print("-" * 80)

for strategy in strategies:
    t_opt = evaluator.optimal_threshold(y_test, xgb_scores, strategy=strategy)
    m = evaluator._metrics_at_threshold(y_test, xgb_scores, t_opt)
    print(
        f"{strategy:<25} {t_opt:>10.4f} {m.precision:>10.4f} "
        f"{m.recall:>10.4f} {m.f1:>10.4f} {m.n_alerts:>8}"
    )

In [ ]:
# Recommended operating point: cost-optimal threshold
t_selected = evaluator.optimal_threshold(y_test, xgb_scores, strategy="cost")
cm_selected = evaluator.cost_matrix(y_test, xgb_scores, threshold=t_selected)

print("Selected operating threshold (cost-optimal):")
print(f"  Threshold:         {t_selected:.4f}")
print(f"  False positives:   {cm_selected.n_false_positives}  (cost: £{cm_selected.total_fp_cost:,.0f})")
print(f"  False negatives:   {cm_selected.n_false_negatives}  (cost: £{cm_selected.total_fn_cost:,.0f})")
print(f"  Total expected cost: £{cm_selected.total_cost:,.0f}")

---
## 8. Feature importance cross-validation

One of the key model validation tasks is checking that the XGBoost model and the logistic regression baseline agree on the most important features. Significant rank disagreement (large `rank_delta`) may indicate:

- **Interaction effects:** XGBoost can capture non-linear interactions between features that linear models miss; this is expected and acceptable
- **Data leakage:** if XGBoost heavily weights a feature that logistic regression de-emphasises, check whether that feature carries implicit label information
- **Overfitting:** if XGBoost weights noisy features highly, it may be overfitting to training artefacts

`compare_with_xgb()` returns a rank-delta DataFrame suitable for inclusion in the model validation report.

In [ ]:
rank_comparison = lr_clf.compare_with_xgb(xgb_clf)
print("Feature importance rank comparison (LR vs XGBoost):")
print(rank_comparison.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = [
    "firebrick" if d > 5 else "steelblue"
    for d in rank_comparison["rank_delta"]
]
ax.barh(rank_comparison["feature"][::-1], rank_comparison["rank_delta"][::-1], color=colors[::-1])
ax.axvline(5, color="firebrick", linestyle="--", alpha=0.5, label="Delta > 5 (review)")
ax.set_xlabel("Rank delta (|LR rank - XGB rank|)")
ax.set_title("Feature importance rank agreement: LR vs XGBoost")
ax.legend()
plt.tight_layout()
plt.show()

high_delta = rank_comparison[rank_comparison["rank_delta"] > 5]
if len(high_delta) > 0:
    print(f"\nFeatures with rank_delta > 5 (requires review in validation report):")
    print(high_delta[["feature", "lr_rank", "xgb_rank", "rank_delta"]].to_string(index=False))
else:
    print("\nNo features with rank_delta > 5. LR and XGBoost are in broad agreement.")

---
## Summary

This notebook has demonstrated the complete `fincrime_ml.fraud` pipeline:

| Stage | Module | Key output |
|---|---|---|
| Synthetic data | `synth_cards` | Labelled transaction DataFrame |
| Feature engineering | `fraud.features` | 16-column feature matrix |
| Imbalance handling | `fraud.imbalance` | Benchmarked strategy choice |
| XGBoost champion | `fraud.models.xgb_classifier` | Fitted model, CV AUC-PR |
| LR challenger | `fraud.models.logistic_baseline` | Fitted baseline, coefficient ranks |
| Explainability | `fraud.explain` | SHAP reason codes per transaction |
| Evaluation | `fraud.evaluation` | Threshold analysis, cost matrix, comparison |

### Regulatory documentation artefacts produced

1. **Champion/challenger AUC-PR comparison** (PRA SS1/23 Section 5): XGBoost versus logistic regression on holdout test set
2. **Threshold sensitivity analysis** (SR 11-7 Section IV): precision, recall, F1, and alert volume at 100 threshold values
3. **Monetary cost matrix** (FCA PROD 2.1): expected GBP cost of false positives and false negatives at the selected operating threshold
4. **Feature importance cross-validation** (PRA SS1/23 Section 4.6): rank-delta comparison between interpretable baseline and XGBoost
5. **Per-transaction reason codes** (FCA SYSC 6.3.1R): SHAP-based explanations suitable for fraud analyst alert review

The selected operating threshold and its monetary justification should be reviewed at each model revalidation cycle.